In [2]:

#Proyecto Base Completo: Temporal Fusion Transformer (TFT) con PyTor

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
import matplotlib.pyplot as plt

# Importaciones de la librería especializada pytorch-forecasting
from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
    GroupNormalizer
)
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data.examples import get_stallion_data

# Establecer semilla para asegurar la reproducibilidad de los resultados de entrenamiento y pesos
pl.seed_everything(42)

RuntimeError: CPU dispatcher tracer already initlized

In [ ]:
# =====================================================================
# BLOQUE 1: CARGA Y PREPROCESAMIENTO
# =====================================================================
print("--- Bloque 1: Carga y Preprocesamiento de Datos ---")

# 1. Carga del dataset Stallion integrado.
# Este dataset contiene datos mensuales de ventas (volumen) para varios SKUs en distintas agencias.
# Es perfecto para series de tiempo multivariante con múltiples entidades (grupos).
data = get_stallion_data()

# 2. Aseguramos un 'time_idx' continuo y de tipo entero.
# Para modelos basados en Deep Learning como TFT, el time_idx debe incrementarse en 1 para cada
# paso de tiempo sucesivo por serie temporal (grupo). Esto le permite al modelo saber la distancia
# temporal exacta entre observaciones y manejar huecos de información si existieran.
# En Stallion este índice ya viene precalculado, pero en datasets reales se debe construir de manera
# continua ordenando cronológicamente las observaciones para cada grupo.
data["time_idx"] = data["time_idx"].astype(int)

# 3. Conversión estricta de variables categóricas.
# Primero se convierten a tipo string para asegurar que los códigos numéricos se interpreten
# textualmente de manera limpia, y luego se convierten formalmente al tipo 'category' de pandas.
# PyTorch Forecasting requiere el tipo 'category' de pandas de manera estricta para mapear y
# tokenizar correctamente estas variables en los módulos de Embedding de la red neuronal.
# Esto previene errores de tipado interno durante la inicialización de los pesos de la red.
categorical_cols = ["agency", "sku", "sku_category", "sub_category", "agency_region", "month"]
for col in categorical_cols:
    data[col] = data[col].astype(str).astype("category")

# Mostramos un resumen rápido del preprocesamiento para verificar la consistencia inicial
print(f"Dimensiones del dataset: {data.shape}")
print(f"Rango del time_idx: {data['time_idx'].min()} a {data['time_idx'].max()}")
print("Variables categóricas preprocesadas con éxito.")

In [ ]:

# =====================================================================
# BLOQUE 2: DEFINICIÓN DEL TIMESERIESDATASET (Núcleo Didáctico)
# =====================================================================
print("\n--- Bloque 2: Definición de la Estructura de Datos (TimeSeriesDataSet) ---")

# Definimos las ventanas temporales para el modelado:
# max_prediction_length: Cuántos pasos en el futuro queremos predecir (6 meses).
# max_encoder_length: Cuántos pasos del pasado utilizaremos como contexto (24 meses).
max_prediction_length = 6
max_encoder_length = 24

# Punto de corte para la partición de entrenamiento (dejamos la última ventana de predicción para validación/testeo)
training_cutoff = data["time_idx"].max() - max_prediction_length

# Creación de la estructura del dataset de entrenamiento.
# Aquí es donde le enseñamos al modelo la semántica de cada variable.
training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",                      # Columna que actúa como índice temporal continuo.
    target="volume",                          # Variable objetivo que queremos predecir.
    group_ids=["agency", "sku"],              # Columnas que definen e identifican de forma única a cada serie temporal individual.
    min_encoder_length=max_encoder_length // 2, # Longitud mínima permitida para el historial (permite secuencias cortas si las hay).
    max_encoder_length=max_encoder_length,    # Longitud máxima de la ventana de contexto histórica.
    min_prediction_length=1,                  # Mínimo de pasos a predecir.
    max_prediction_length=max_prediction_length, # Máximo de pasos a predecir.
    
    # -----------------------------------------------------------------
    # variables estáticas categóricas (STATIC_CATEGORICALS)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Son características categóricas que NO cambian con el tiempo para un grupo dado (ej: una combinación de agencia y SKU).
    # El modelo las procesa a través de embeddings compartidos estáticos que guían la inicialización
    # del estado de la red LSTM (Variable Selection Network - VSN estática). Esto permite transferir
    # conocimiento de patrones entre series que comparten atributos estáticos similares (ej: el mismo SKU o la misma Región).
    static_categoricals=["agency", "sku", "sku_category", "sub_category", "agency_region"],
    
    # -----------------------------------------------------------------
    # variables estáticas reales (STATIC_REALS)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Atributos numéricos continuos que no cambian en el tiempo para el mismo grupo.
    # En este caso, son datos demográficos o socioeconómicos fijos del año 2017 para cada agencia.
    # Ayudan a escalar las expectativas del volumen de ventas según la población o el poder adquisitivo del área.
    static_reals=["avg_population_2017", "avg_yearly_household_income_2017"],
    
    # -----------------------------------------------------------------
    # variables categóricas variables conocidas en el futuro (TIME_VARYING_KNOWN_CATEGORICALS)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Cambian a lo largo del tiempo, pero sabemos su valor exacto para los pasos de tiempo futuros.
    # El mes ("month") es una variable estacional calendario perfectamente determinista y conocida con antelación en el futuro.
    time_varying_known_categoricals=["month"],
    
    # -----------------------------------------------------------------
    # variables reales variables conocidas en el futuro (TIME_VARYING_KNOWN_REALS)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Atributos numéricos que cambian temporalmente y de los cuales conocemos sus valores futuros de forma exacta.
    # El "time_idx" es secuencial y perfectamente determinista. Los precios y descuentos ("price_regular", 
    # "price_actual", "discount", "discount_in_percent") son conocidos de antemano debido a la planificación 
    # comercial y promocional. El TFT los aprovecha tanto en el encoder (pasado) como en el decoder (futuro)
    # para aprender el impacto directo de las promociones futuras sobre las ventas.
    time_varying_known_reals=[
        "time_idx",
        "price_regular",
        "price_actual",
        "discount",
        "discount_in_percent"
    ],
    
    # -----------------------------------------------------------------
    # variables reales variables desconocidas en el futuro (TIME_VARYING_UNKNOWN_REALS)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Atributos continuos cuyo valor futuro es una incógnita en el momento de realizar la predicción.
    # La variable objetivo "volume" y su transformación "log_volume" cambian dinámicamente y solo se conocen
    # en el pasado. El volumen global de la industria ("industry_volume"), el volumen de refrescos ("soda_volume")
    # y la temperatura máxima media ("avg_max_temp") son métricas del entorno observadas históricamente, 
    # pero imposibles de saber de forma exacta en el horizonte de predicción. Por lo tanto, el TFT solo 
    # los procesará en la ventana del encoder (historial) para entender el contexto dinámico reciente.
    time_varying_unknown_reals=[
        "volume",
        "log_volume",
        "industry_volume",
        "soda_volume",
        "avg_max_temp"
    ],
    
    # -----------------------------------------------------------------
    # normalizador del target (TARGET_NORMALIZER)
    # -----------------------------------------------------------------
    # ¿POR QUÉ AQUÍ?: Las ventas por combinación de agencia/SKU varían enormemente en escala (magnitudes diferentes).
    # Usamos GroupNormalizer que normaliza cada serie temporal de forma independiente basada en sus propias estadísticas
    # de grupo. Elegimos 'softplus' como transformación para evitar que los valores normalizados escalados 
    # devuelvan predicciones negativas, ya que el volumen de ventas siempre debe ser no negativo (>= 0).
    target_normalizer=GroupNormalizer(
        groups=["agency", "sku"], 
        transformation="softplus"
    ),
    
    # -----------------------------------------------------------------
    # características adicionales integradas del TFT
    # -----------------------------------------------------------------
    add_relative_time_idx=True,  # Agrega una variable temporal relativa (-24 a +6) para ayudar al mecanismo de atención.
    add_target_scales=True,      # Agrega la media y desviación del target como features estáticos (ayuda con la escala de forma robusta).
    add_encoder_length=True,     # Permite al modelo saber de qué tamaño es la historia de entrada de forma explícita.
)

print("Estructura de TimeSeriesDataSet creada correctamente.")



In [ ]:

# =====================================================================
# BLOQUE 3: DATALOADERS
# =====================================================================
print("\n--- Bloque 3: Creación de DataLoaders ---")

# 1. Dataset de validación.
# Usamos 'TimeSeriesDataSet.from_dataset' para clonar la estructura exacta, normalizaciones,
# mapeos de categorías y parámetros del conjunto de entrenamiento de forma consistente.
# Con 'predict=True', le indicamos al dataset que solo devuelva el ÚLTIMO corte temporal de cada 
# serie (un único ejemplo por grupo), que es exactamente lo que necesitamos para evaluar el 
# rendimiento predictivo final en la ventana futura real sin redundancias ni solapamientos.
validation = TimeSeriesDataSet.from_dataset(
    training, 
    data, 
    predict=True, 
    stop_randomization=True
)

# 2. Creación de los DataLoaders de PyTorch.
# batch_size: Cantidad de ejemplos procesados en paralelo. Ajustable según hardware.
# num_workers: 0 se prefiere por robustez multiplataforma (especialmente en Windows).
# train_dataloader: Habilitamos 'shuffle=True' para desordenar aleatoriamente los lotes durante el 
# entrenamiento, garantizando que el optimizador no aprenda sesgos basados en el orden de las series.
# val_dataloader: No necesita barajarse ('train=False') para mantener la coherencia de evaluación de validación.
batch_size = 128
train_dataloader = training.to_dataloader(
    train=True, 
    batch_size=batch_size, 
    num_workers=0
)
val_dataloader = validation.to_dataloader(
    train=False, 
    batch_size=batch_size * 10, 
    num_workers=0
)

print(f"DataLoader de entrenamiento: {len(train_dataloader)} lotes creados.")
print(f"DataLoader de validación: {len(val_dataloader)} lotes creados.")



In [ ]:

# =====================================================================
# BLOQUE 4: INSTANCIACIÓN DEL MODELO
# =====================================================================
print("\n--- Bloque 4: Instanciación del Modelo Temporal Fusion Transformer (TFT) ---")

# Creamos el modelo TFT utilizando 'from_dataset' para mapear automáticamente la arquitectura de la red
# con el número de variables de entrada, dimensiones de categorías y estructura del TimeSeriesDataSet.
#
# Explicación de los Hiperparámetros Básicos de TFT:
# - hidden_size: Controla la capacidad de representación de la red (número de neuronas en las capas LSTM
#   y de atención). Se establece en 16 para este caso de prueba rápido para evitar sobreajuste en el dataset Stallion.
# - attention_head_size: Número de cabezas de atención en el módulo de auto-atención auto-regresivo. Permite
#   al TFT capturar diferentes dependencias a largo plazo simultáneamente.
# - learning_rate: Tasa de aprendizaje inicial para el optimizador Adam.
# - loss: QuantileLoss. A diferencia de las redes tradicionales que predicen un único punto (ej: MSE),
#   el TFT está diseñado para la previsión de intervalos probabilísticos. El QuantileLoss optimiza la predicción
#   para múltiples cuantiles (por defecto: p10, p50, p90, etc.), lo cual es indispensable para la toma de decisiones con 
#   incertidumbre (por ejemplo, p10 representa un escenario pesimista de demanda y p90 el escenario más optimista).
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,           # Tasa de aprendizaje para la actualización de pesos.
    hidden_size=16,               # Dimensión de representación oculta para variables y estados temporales.
    attention_head_size=4,        # Cabezas del mecanismo de auto-atención multi-cabeza temporal.
    dropout=0.1,                  # Ratio de regularización dropout para mitigar el sobreajuste.
    loss=QuantileLoss(),          # Función de pérdida basada en cuantiles por defecto (p02, p10, p25, p50, p75, p90, p98).
    reduce_on_plateau_patience=4, # Reduce automáticamente la tasa de aprendizaje si el val_loss se estanca.
)

print(f"Modelo TFT instanciado correctamente.")
print(f"Número total de parámetros entrenables: {sum(p.numel() for p in tft.parameters() if p.requires_grad):,}")



In [ ]:

# =====================================================================
# BLOQUE 5: ENTRENAMIENTO
# =====================================================================
print("\n--- Bloque 5: Configuración del Entrenador y Entrenamiento ---")

# Configuración del Trainer de PyTorch Lightning.
# Para este proyecto base enfocado en testeo rápido y didáctico, se configura a 3 épocas.
# - max_epochs: Establecido en 3 para garantizar un ciclo de entrenamiento ultra-rápido de validación técnica.
# - accelerator & devices: "auto" permite a PyTorch Lightning seleccionar de manera óptima GPU (CUDA) si está
#   disponible, o CPU en su defecto, haciendo el código portable a cualquier hardware.
# - gradient_clip_val: Limitador del valor del gradiente para prevenir el desvanecimiento o explosión del
#   gradiente durante el entrenamiento de las componentes recurrentes (LSTMs).
# - callbacks:
#   * EarlyStopping: Detiene el entrenamiento prematuramente si la pérdida de validación deja de mejorar,
#     protegiendo contra el sobreajuste y ahorrando recursos informáticos.
#   * LearningRateMonitor: Registra la evolución de la tasa de aprendizaje en las herramientas de visualización (TensorBoard).

early_stop_callback = EarlyStopping(
    monitor="val_loss", 
    min_delta=1e-4, 
    patience=10, 
    verbose=False, 
    mode="min"
)
lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=3,
    accelerator="auto",
    devices="auto",
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, lr_logger],
)

print("Iniciando el entrenamiento...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
print("Entrenamiento completado de forma exitosa.")



In [ ]:

# =====================================================================
# BLOQUE 6: EVALUACIÓN Y VISUALIZACIÓN
# =====================================================================
print("\n--- Bloque 6: Evaluación y Visualización de Resultados ---")

# 1. Extracción de predicciones probabilísticas.
# 'mode="quantiles"' le indica al modelo que debe devolver las matrices tridimensionales correspondientes
# a cada cuantil configurado en el QuantileLoss.
# 'return_x=True' nos devuelve las características de entrada completas, fundamentales para que la función
# de graficado sepa qué parte de la serie temporal pertenece al pasado (encoder) y cuál al futuro (decoder).
predictions = tft.predict(val_dataloader, mode="quantiles", return_x=True)

# 2. Generación del gráfico comparativo.
# Usamos el método nativo 'plot_prediction' de PyTorch Forecasting, diseñado específicamente para TFT.
# Este método realiza un renderizado excepcional:
#   - Línea gris/negra discontinua: Valores históricos observados (encoder).
#   - Línea azul discontinua: El valor real en el horizonte de predicción (decoder actual).
#   - Línea de predicción p50 (mediana): Representa la predicción central (esperada).
#   - Sombra o bandas de confianza: Delimitan el intervalo probabilístico entre los cuantiles seleccionados (p10 a p90),
#     mostrando visualmente la incertidumbre y el nivel de confianza que tiene el modelo sobre su predicción.
fig, ax = plt.subplots(figsize=(12, 6))

# Graficamos la predicción para el primer elemento de la validación (idx=0)
tft.plot_prediction(
    predictions.x, 
    predictions.output, 
    idx=0, 
    add_loss_to_title=True, 
    ax=ax
)

# Configuración del título y etiquetas en español para mayor claridad didáctica
ax.set_title("Predicción TFT: Valor Real vs Predicciones Probabilísticas (Cuantiles p10-p50-p90)", fontsize=14, pad=15)
ax.set_xlabel("Paso del Tiempo (time_idx)", fontsize=11)
ax.set_ylabel("Volumen de Ventas (Target)", fontsize=11)

# Guardar el gráfico en un archivo de imagen local.
# Esta es una mejor práctica para entornos de desarrollo en la nube o CLI, asegurando que el proceso
# termine de manera limpia sin colgarse esperando interacción de ventana GUI.
plot_path = "tft_prediction_plot.png"
fig.savefig(plot_path, bbox_inches="tight", dpi=300)
plt.close(fig)

print(f"¡Proceso finalizado con éxito!")
print(f"El gráfico de evaluación se ha guardado en: {os.path.abspath(plot_path)}")
print("Este archivo de imagen muestra los valores reales de validación frente a la predicción p50 y las bandas de incertidumbre p10-p90.")
